# Current workflows at the original notebook address

This notebook is refreshed from the maintained workflow examples: [Classify objects from measured features](../04_classify_machine_learning.ipynb) · [Estimate genotype-phenotype associations](../06_regression.ipynb). Configure each stage's inputs before running it. Saved settings and API calls below use current names and defaults.

[All workflows](../README.md) · [Workflow browser](../00_workflow_browser.ipynb)

# Classify objects from measured features

**Purpose.** Fit a supervised tabular classifier to per-object measurements and apply it to the full dataset.

**Recommended use.** Use when morphology, intensity, or spatial measurements provide an interpretable representation of the phenotype.

**Primary outputs.** Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

**Desktop route.** Classify → Machine Learning

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.ml.generate_ml_scores`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.generate_ml_scores)

```python
generate_ml_scores(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.ml import generate_ml_scores

## 3. Settings and API reference

Start with the essential settings below. Expand a reference group to read its parameters. Advanced and organelle code cells start collapsed in JupyterLab and Notebook 7: click their ellipsis to edit them. Run all settings cells before the analysis cell, including collapsed cells. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the spaCR source used to generate this notebook, keeping it aligned with the public API.

### [`spacr.ml.generate_ml_scores`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.generate_ml_scores)


<details>
<summary>Labels &amp; Classes (6 settings)</summary>

#### Labels & Classes

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`dataset_mode`** *(optional)* — (str) - How training classes are defined: 'metadata' splits crops by well metadata, 'annotation' by the values in one or more annotation columns of png_list. Either way the classes themselves are set in the Classes editor, which names a column and a value per class. A settings file written before the 'measurement' basis was removed still loads: it is read as 'annotation', which is what its threshold rules resolved to after writing their label column. Any other value aborts and returns no dataset. Default 'metadata'.
- **`location_column`** *(optional)* — (str) - Metadata column searched for positive_control_id and negative_control_id values when labelling rows for machine-learning training, normally 'columnID' or 'rowID'. Set 'rowID' when controls are arranged along plate rows instead of columns. annotation_column overrides this setting when specified. Default 'columnID'.
- **`positive_control_id`** *(conditionally required)* — (str) - Identifier of the positive-control class. In ML screening it is the value in location_column (e.g. 'c2') whose objects are labelled class 1 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '239740') matched against coefficient names to tag them 'pc' in the results and volcano plot. Defaults 'c2' and '239740' respectively.
- **`negative_control_id`** *(conditionally required)* — (str) - Identifier of the negative-control class. In ML screening it is the value in location_column (e.g. 'c1') whose objects are labelled class 0 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '233460') matched against coefficient names to tag them 'nc' in the results and volcano plot. Defaults 'c1' and '233460' respectively.
- **`annotation_column`** *(conditionally required)* — (str) - Integer column in the png_list table that stores manual class labels. The Annotate app adds it with ALTER TABLE if it is absent and writes labels to it. This column provides the reference labels when dataset_mode is 'annotation' and is the fallback when annotation_columns is unset. Supplying it while dataset_mode is unset also selects annotation mode for compatibility with older settings files. Default None.

</details>

<details>
<summary>Feature Preparation (7 settings)</summary>

#### Feature Preparation

- **`channel_of_interest`** *(optional)* — (int, list, or str) - Measurements available to the model. Specify one channel to train on that channel alone, multiple channels to use their combination of measurements, or 'shape' to use outline-derived features. An empty value includes every measurement. A colocalisation feature is associated with both measured channels, so selecting either channel also includes their shared colocalisation features. This setting also selects the channel used for recruitment measurements. Default 3 in machine-learning steps and 1 or 2 elsewhere.
- **`exclude`** *(optional)* — (str or list) - Names of measurement columns to drop from the feature set before UMAP embedding or ML training, applied after the channel_of_interest selection. Use it to remove features that leak the label or swamp the embedding. It does not filter database rows; use exclude_rows for that. Default None keeps every feature.
- **`nuclei_limit`** *(optional)* — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do not pass False: it is interpreted as 0 and removes every cell, leaving an empty analysis rather than raising an error. Default None. Merged Classifier starts at True and Recruitment starts at 1, so both initially retain only single-nucleus cells. Replication starts at 10.
- **`pathogen_limit`** *(optional)* — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.
- **`remove_highly_correlated_features`** *(optional)* — (bool) - In the machine-learning feature table, drop any feature whose absolute Pearson correlation with an already-kept feature exceeds 0.95, applied after the channel_of_interest filter. Leave it on so redundant measurements do not split importance scores and slow fitting; turn it off only when you need every original column. Default True. Note the UMAP path uses remove_highly_correlated instead.
- **`remove_low_variance_features`** *(optional)* — (bool) - Drop numeric features whose variance across objects falls below 0.01 before model fitting -- near-constant columns that carry no discriminative signal but still cost time and dilute importance rankings. Turn it off only when your features live on a very small numeric scale, where genuine signal can fall under that fixed cut-off. Default True.
- **`min_cells_per_well`** *(optional)* — (int) - Wells with fewer than this many cells are dropped. In a regression it is scored objects and the well is left out of the fit; in the machine-learning screen it is measured cells and the well is left out of the plate heatmap, whose pivot is then filled with 0, so an excluded well renders at the bottom of the colour scale rather than blank. Raising it removes noisy, sparsely imaged wells at the cost of power. Set 0 to switch it off. Default 100 for a regression, 25 for the screen.

</details>

<details>
<summary>Plate &amp; Batch Correction (8 settings)</summary>

#### Plate & Batch Correction

- **`batch_correction`** *(optional)* — (str) - Plate/batch correction applied before Image UMAP, ML screen classification or phenotype regression. 'none' leaves measurements alone; 'center' removes each plate's mean shift; 'zscore' aligns plate means and variances; 'robust_zscore' uses median/MAD and tolerates outliers; 'combat' models the batch effect while protecting the terms named in batch_covariate_column. Correct when plates were stained or imaged separately; leave off when they were not, since every method removes real signal that happens to align with plate. See spacr.batch_correction.correct_batch_effects. Default 'none'.
- **`batch_column`** *(optional)* — (str) - Metadata column that identifies independent acquisition batches, normally 'plateID'. Every analyzed row must have a value and at least batch_min_samples rows must occur in each batch. Use an acquisition date or instrument ID only if that is the nuisance source you intend to remove. Default 'plateID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_column`** *(optional)* — (str or None) - Metadata column containing reference-control labels for control_center, normally 'columnID' for plate controls. It is ignored by center, zscore, robust_zscore, and none. Blank follows col_to_compare in Image UMAP or location_column in Classify (ML); regression defaults to 'columnID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_values`** *(optional)* — (str, number, list or None) - Reference/negative-control value(s) in batch_control_column used by control_center. Each plate needs at least batch_min_samples matching rows. Image UMAP falls back to neg and Classify (ML) to negative_control_id when this field is blank; regression requires an explicit value. Default varies by module. API: spacr.batch_correction.correct_batch_effects.
- **`batch_covariate_column`** *(optional)* — (str, list or None) - Metadata column(s) naming the biological effects ComBat must preserve, for example 'condition' or 'condition,timepoint'. ComBat estimates the batch effect from residuals after fitting these terms, so unlisted effects may be removed with the plate effect. Include every treatment effect that must remain in the corrected data. See spacr.batch_correction.correct_batch_effects. Default None.
- **`batch_combat_mean_only`** *(optional)* — (bool) - True corrects only the additive batch shift and leaves each batch's scale alone. Use it when the plates differ in level but not in spread, or when a batch has too few rows for a stable variance estimate. False (the default) corrects both location and scale, which is standard ComBat. Ignored by every method other than combat. API: spacr.batch_correction.correct_batch_effects.
- **`batch_min_samples`** *(optional)* — (int) - Minimum number of rows required in every batch, and minimum matching reference controls per batch for control_center. Correction stops with an actionable error below this threshold because a one- or two-object plate estimate is unstable. Default 3. API: spacr.batch_correction.correct_batch_effects.
- **`batch_missing_control`** *(optional)* — (str) - Policy when control_center cannot find enough reference controls on a plate: 'error' stops rather than silently mixing corrected and raw plates; 'skip' leaves that plate unchanged and records a warning. Default 'error'. API: spacr.batch_correction.correct_batch_effects.

</details>

<details>
<summary>Classifier &amp; Validation (7 settings)</summary>

#### Classifier & Validation

- **`model_type_ml`** *(optional)* — (str) - Classifier fitted by ml_analysis to separate positive- from negative-control wells and rank per-object features by permutation importance. Options are xgboost (default), lightgbm, catboost, random_forest, extra_trees, gradient_boosting, logistic_regression, svm and mlp; lightgbm and catboost require their optional packages. reg_alpha, reg_lambda and learning_rate affect only boosted models; logistic_regression provides a linear reference model.
- **`n_estimators`** *(optional)* — (int) - Number of trees or boosting rounds in the tabular ML classifier - n_estimators for RandomForest/ExtraTrees/XGBoost/LightGBM, iterations for CatBoost, max_iter for HistGradientBoosting. More rounds keep improving fit up to a plateau while training time grows linearly; boosted models can overfit past it. Default 1000.
- **`learning_rate`** *(optional)* — (float) - Initial optimizer step size. Values that are too high may prevent convergence; values that are too low may slow convergence or converge to a suboptimal solution. A value near 1e-3 is commonly used for training from random initialization, while 1e-4 to 1e-5 is appropriate for fine-tuning ImageNet weights (init_weights=True). The selected schedule modifies this initial value during training. Default 0.001.
- **`test_size`** *(optional)* — (float) - Fraction of labelled single-object rows reserved as the test split for the tabular machine-learning classifier; the remainder trains the model. Increasing the value reduces uncertainty in the accuracy estimate but leaves fewer labelled rows for training. Valid range 0-1; default 0.2 (20% test).
- **`cross_validation`** *(optional)* — (bool) - Score the feature-based classifier (Classify, 'ml' family) with class-stratified cross-validation instead of a single train/test split: up to 5 folds, whole cv_group_by groups per fold, so every control object receives an out-of-fold prediction. Each fold's optimal probability threshold is computed and printed when verbose is on; predictions are not re-thresholded. Gives a less variable accuracy estimate on small control sets, at roughly 5x the training time. Default True.
- **`reg_alpha`** *(optional)* — (float) - L1 penalty on leaf weights for the gradient-boosted classifier (XGBoost and LightGBM; ignored by the other model_type_ml choices). Raising it drives more leaf weights to exactly zero, shrinking the model and its effective feature set - raise it when training accuracy far exceeds test accuracy. Any value &gt;= 0. Default 0.1.
- **`reg_lambda`** *(optional)* — (float) - L2 penalty on leaf weights for the gradient-boosted classifier (XGBoost, LightGBM, and CatBoost's l2_leaf_reg). Raising it shrinks all weights smoothly rather than zeroing them, damping the influence of any single feature and curbing overfitting, at the risk of underfitting if pushed too far. Any value &gt;= 0. Default 1.0.

</details>

<details>
<summary>Feature Selection &amp; Importance (3 settings)</summary>

#### Feature Selection & Importance

- **`prune_features`** *(optional)* — (bool) - Before training, keep only the top_features columns with the highest ANOVA F-score against the control labels (sklearn SelectKBest with f_classif). Speeds up fitting and can curb overfitting on small control sets, but discards features the model might have used and scores each feature in isolation, ignoring interactions. Default False.
- **`top_features`** *(optional)* — (int) - Feature cap in the ML screen analysis: how many rows the feature-importance and permutation-importance bar plots show, and how many top-ranked features the SHAP refit and its summary plot use. It is also the k of the SelectKBest pruning applied before the model is fitted, but only when prune_features is True - with prune_features at its default False the classifier trains on every feature and this is reporting/SHAP scope only. Raise for a fuller picture, lower for readable plots. Default 30.
- **`n_repeats`** *(optional)* — (int) - Number of random shuffles per feature when computing permutation importance for the ML classifier. More repeats reduce uncertainty in the importance ranking but require an additional prediction pass per feature and repeat. Default 10; use 3-5 for a faster preliminary assessment of wide feature tables.

</details>

<details>
<summary>Plots &amp; Heatmaps (4 settings)</summary>

#### Plots & Heatmaps

- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') preserve the relative visibility of intensity differences; 'gray' resembles the raw single-channel microscope image. Any registered matplotlib name is accepted, with an '_r' suffix to reverse it. Default 'inferno' for image plots and 'viridis' for plate heatmaps.
- **`heatmap_feature`** *(optional)* — (str) - Numeric column that is aggregated per well and color-mapped in the plate heatmap after ML scoring, e.g. 'predictions' for the classifier score or 'recruitment' for the pathogen/cytoplasm intensity ratio. Must be a numeric column of the scored dataframe or the run raises ValueError listing the valid names. Default 'predictions'.
- **`grouping`** *(optional)* — (str) - How per-object values collapse to one number per well in the plate heatmap: 'mean' averages heatmap_feature over the objects in a well, 'sum' totals them, 'count' ignores the feature and colors wells by object count. Use 'count' to spot uneven seeding or dropout, 'mean' for phenotype strength. Default 'mean'; any other value raises ValueError.
- **`min_max`** *(optional)* — (str) - Color limits for the plate heatmap: 'allq' scales to the 2nd-98th percentile of well values so a handful of extreme wells cannot flatten the rest, 'all' scales to the true min and max. A two-element list is also accepted, where floats are read as quantiles and integers as absolute vmin/vmax. Default 'allq'.

</details>

<details>
<summary>Runtime &amp; Reliability (3 settings)</summary>

#### Runtime & Reliability

- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression. Invasion and Replication also start with console detail disabled.
- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`ram_guard`** *(optional)* — (bool) - Keep worker pools from filling RAM. Before any module starts its workers, spaCR estimates each worker's memory from one input (a field, mask, crop batch or table) and lowers n_jobs to the count that leaves 12.5% of RAM free, printing a warning; the GUI asks first. Off keeps your n_jobs, and Measure fields still wait while free RAM is below that reserve. Default True.

</details>

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Labels & Classes
    # Required settings
    'src': 'path',
    # Conditionally required settings
    'positive_control_id': 'c2',
    'negative_control_id': 'c1',
    'annotation_column': None,

    # Runtime & Reliability
    # Optional settings
    'n_jobs': -1,
}

In [ ]:
settings.update({
    # Labels & Classes
    # Optional settings
    'dataset_mode': 'metadata',
    'location_column': 'columnID',

    # Feature Preparation
    # Optional settings
    'channel_of_interest': 3,
    'exclude': None,
    'nuclei_limit': True,
    'pathogen_limit': 3,
    'remove_highly_correlated_features': True,
    'remove_low_variance_features': True,
    'min_cells_per_well': 25,

    # Plate & Batch Correction
    # Optional settings
    'batch_correction': 'none',
    'batch_column': 'plateID',
    'batch_control_column': None,
    'batch_control_values': None,
    'batch_covariate_column': None,
    'batch_combat_mean_only': False,
    'batch_min_samples': 3,
    'batch_missing_control': 'error',

    # Classifier & Validation
    # Optional settings
    'model_type_ml': 'xgboost',
    'n_estimators': 1000,
    'learning_rate': 0.001,
    'test_size': 0.2,
    'cross_validation': True,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,

    # Feature Selection & Importance
    # Optional settings
    'prune_features': False,
    'top_features': 30,
    'n_repeats': 10,

    # Plots & Heatmaps
    # Optional settings
    'cmap': 'viridis',
    'heatmap_feature': 'predictions',
    'grouping': 'mean',
    'min_max': 'allq',

    # Runtime & Reliability
    # Optional settings
    'verbose': True,
    'ram_guard': True,
})

In [ ]:
generate_ml_scores(settings)

## Outputs and next steps

Per-object predictions, validation metrics, feature importance, and SHAP summaries when supported.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)

# Estimate genotype-phenotype associations

**Purpose.** Estimate guide- or gene-level effects by modelling phenotype scores as a function of guide abundance.

**Recommended use.** Use after phenotype scores and matched guide-count tables have been generated for the same experimental wells.

**Primary outputs.** Effect estimates, uncertainty or resampling support, adjusted significance values, diagnostic plots, and ranked findings.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.ml.perform_regression`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.perform_regression)

```python
perform_regression(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.ml import perform_regression

## 3. Settings and API reference

Start with the essential settings below. Expand a reference group to read its parameters. Advanced and organelle code cells start collapsed in JupyterLab and Notebook 7: click their ellipsis to edit them. Run all settings cells before the analysis cell, including collapsed cells. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the spaCR source used to generate this notebook, keeping it aligned with the public API.

### [`spacr.ml.perform_regression`](https://einarolafsson.github.io/spacr/api/spacr/ml/index.html#spacr.ml.perform_regression)


<details>
<summary>Input Tables (9 settings)</summary>

#### Input Tables

- **`score_data`** *(required)* — (str or list) - CSV(s) of per-object or per-well phenotype scores, typically from generate_ml_scores. Each must contain dependent_variable. Pass one path per plate, position-aligned with plates_score. The score filename does not name the output folder: runs go under src/results, named for the inference or regression kind and then suffixed _1, _2, and so on. When src is unset, results is created beside the first count_data file. Default 'list of paths'.
- **`count_data`** *(required)* — (str or list) - CSV(s) of per-well gRNA read counts from the sequencing step (unique_combinations.csv); each must contain grna, count, rowID and columnID columns or the run raises ValueError. These are the regression's independent variable. Pass one path per plate, position-aligned with plates_count; results are written under the first file's folder. Default 'list of paths', a placeholder that must be replaced; the barcode QC module defaults this key to 'path to unique_combinations.csv'.
- **`paired_data`** *(required)* — (list of dicts) - Regression input table: each row explicitly pairs one score CSV with one count CSV. Plate identity comes from both files when they agree, from the partner when only one declares plateID, or from the row order when neither does. A conflict is refused. Legacy score_data/count_data lists are migrated positionally and logged. Default [].
- **`metadata_files`** *(optional)* — (list) - Gene-annotation CSVs, each with a 'Gene ID' column, that are joined onto the regression results by gene, writing an extra results CSV per file. These are gene tables, not plate/well metadata. When toxo is True the order matters: index 0 is read as the ME49 transcription table and index 1 as the GT1 phenotype table. Default [].
- **`count_grna_column`** *(optional)* — (str) - Name of the column in the count CSV containing the guide identifier. Set this value to the column produced by the sequencing pipeline. Default 'grna'.
- **`count_value_column`** *(optional)* — (str) - Name of the column in the count CSV holding the read count for one guide in one well; it becomes the per-well fraction the fraction_threshold sweep works on. Set it when the file names this column differently, for example 'reads' or 'n'. Default 'count'.
- **`independent_variable_layout`** *(optional)* — (str) - Shape of the independent-variable/count table: 'long' means one row per well and guide, 'wide' means one row per well with one guide per column, and 'auto' detects long from count_grna_column plus count_value_column and otherwise treats the numeric non-metadata columns as guides. Wide input is melted losslessly before filtering. Default 'auto'.
- **`wide_predictor_columns`** *(optional)* — (list) - Guide columns in a wide independent-variable table. Leave empty to use all numeric columns other than plate/well metadata; list them explicitly when the table contains additional numeric metadata. Ignored for long input. Default [].
- **`src`** *(optional)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.

</details>

<details>
<summary>Controls &amp; Filters (17 settings)</summary>

#### Controls & Filters

- **`positive_control_id`** *(optional)* — (str) - Identifier of the positive-control class. In ML screening it is the value in location_column (e.g. 'c2') whose objects are labelled class 1 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '239740') matched against coefficient names to tag them 'pc' in the results and volcano plot. Defaults 'c2' and '239740' respectively.
- **`negative_control_id`** *(optional)* — (str) - Identifier of the negative-control class. In ML screening it is the value in location_column (e.g. 'c1') whose objects are labelled class 0 for training; in gRNA regression it is a gene/gRNA ID substring (e.g. '233460') matched against coefficient names to tag them 'nc' in the results and volcano plot. Defaults 'c1' and '233460' respectively.
- **`positive_control_wells`** *(optional)* — (list or str) - Wells containing only the positive control, e.g. ['c2']. Accepts rows (r1), columns (c1), or individual wells (A01); the Plate button selects them from a map. Pure controls are calibration references rather than screen observations, so they are excluded from the regression. They define the positive endpoint for mixed-ratio calibration and should be identified from the plate design. Default None.
- **`negative_control_wells`** *(optional)* — (list or str) - Wells containing only the negative control, e.g. ['c1']. Accepts the same row, column, and well notation as positive_control_wells. These wells are excluded from the regression and define the negative endpoint for mixed-ratio calibration. Default None.
- **`mixed_control_wells`** *(optional)* — (list or str) - Wells holding a known mixture of the positive and negative controls, e.g. ['c3']. Removed from the regression like the other two. Per-cell identities in these wells are unknown, but the aggregate proportions are known from sequencing, so annotation methods can be scored on real rather than simulated cells. Default None.
- **`exclude_grnas`** *(optional)* — (list or str) - gRNA or gene identifiers known not to occur in cells, such as primer or plasmid carry-over. These sequences are removed before guide fractions are calculated, and retained guides are renormalized. This differs from background subtraction, which corrects spurious reads assigned to a real guide. A gene identifier matches all of its guides. Default None.
- **`nontargeting_control_grnas`** *(optional)* — (list) - Non-targeting control gRNA identifiers. Their coefficients set the volcano effect-size cutoff: abs(median(control coefficients)) + threshold_multiplier × spread, with threshold_method selecting the spread estimator. A wider control distribution raises the cutoff; None disables it. Default ['000000'] names the non-cutting control gene, which spaCR resolves to all associated guides in the loaded library. Individual guide identifiers also work, with or without the organism prefix.
- **`filter_column`** *(optional)* — (str) - Metadata column used to drop control wells before regression: every row whose value appears in filter_value is removed from both the score data and the read counts. Use 'columnID' (default) when controls sit in plate columns, 'rowID' when they sit in rows. In annotate_filter_vision it instead names the score column thresholded by upper_threshold/lower_threshold.
- **`filter_value`** *(optional)* — (list) - Values of filter_column whose rows are removed - not kept - before regression, normally the control columns; default ['c1','c2','c3']. Dropping them stops control wells from dominating the gene and gRNA fits. Only list values take effect: a bare string is silently ignored and nothing is filtered.
- **`min_cells_per_well`** *(optional)* — (int) - Wells with fewer than this many cells are dropped. In a regression it is scored objects and the well is left out of the fit; in the machine-learning screen it is measured cells and the well is left out of the plate heatmap, whose pivot is then filled with 0, so an excluded well renders at the bottom of the colour scale rather than blank. Raising it removes noisy, sparsely imaged wells at the cost of power. Set 0 to switch it off. Default 100 for a regression, 25 for the screen.
- **`min_observations_per_hit`** *(optional)* — (int) - Observation count a significant hit must strictly exceed to appear in results_significant_filtered.csv: gRNA hits need n_grna &gt; min_observations_per_hit, gene hits need n_gene &gt; min_observations_per_hit. The unfiltered hit list is still written alongside it. Raise it to drop hits resting on one or two wells. Default 0, which filters nothing.
- **`fraction_threshold`** *(optional)* — (float or None) - Minimum relative abundance, 0-1, that a gRNA must reach within a well's total read count to be retained. Increasing it removes low-abundance and bleed-through gRNAs and reduces the mean number of gRNAs per well; if set too high, every row is removed and the run raises an error. Use None to select automatically the cutoff that yields target_unique_count gRNAs per well. Default None. Regression starts at the reproducible fixed cutoff 0.02; enter None only when the automatic sweep is intended.
- **`calibrate_fraction_threshold`** *(optional)* — (bool) - Estimate fraction_threshold from control wells instead of using the configured value. The sweep recomputes per-well fractions across candidate cutoffs and selects the cutoff with greatest imaging-sequencing agreement. The plate design must identify pure control wells independently; selecting controls by the measured fraction would be circular. Default False.
- **`normalise_fraction`** *(optional)* — (bool) - Divide a gRNA's fraction by the sum of the fractions that remain in its well after fraction_threshold, before deciding how many cells it is given. On, a gRNA's share is measured against what survived the threshold; off, it is measured against every read the well produced, including those the threshold removed. The two differ whenever the threshold removes anything: normalising raises every surviving share, and by more the more was removed. Default True.
- **`target_unique_count`** *(optional)* — (int) - Desired mean number of distinct gRNAs per well. spaCR evaluates 1000 read-fraction thresholds, selects the threshold whose per-well mean unique-gRNA count has the smallest absolute difference from this value, and discards every gRNA call below that fraction. Decrease it for a stricter well assignment or increase it to retain more gRNAs per well. Default 5.
- **`tolerance`** *(optional)* — (int or float) - How close a subsampled well mean has to be to the full-well mean before minimum_cell_simulation calls that sample size sufficient, which is what sets min_cells_per_well when you leave it None. An int is read as a percentage (2 means 2%), a float as a fraction (0.02 means the same); anything else raises ValueError. Tighten it toward 0.01 to demand more cells per well and drop more wells, loosen it to 0.05 to keep sparse wells at the cost of noisier per-well scores. Default 0.02.
- **`outlier_detection`** *(optional)* — (bool) - After building the regression table, drop gRNAs whose well count falls outside 1.5x the 5th-95th percentile spread, then recompute the per-gRNA tables. This removes gRNAs present in implausibly few or many wells that would otherwise dominate coefficients; disable it if your library is deliberately uneven. Default True.

</details>

<details>
<summary>Plate &amp; Batch Correction (8 settings)</summary>

#### Plate & Batch Correction

- **`batch_correction`** *(optional)* — (str) - Plate/batch correction applied before Image UMAP, ML screen classification or phenotype regression. 'none' leaves measurements alone; 'center' removes each plate's mean shift; 'zscore' aligns plate means and variances; 'robust_zscore' uses median/MAD and tolerates outliers; 'combat' models the batch effect while protecting the terms named in batch_covariate_column. Correct when plates were stained or imaged separately; leave off when they were not, since every method removes real signal that happens to align with plate. See spacr.batch_correction.correct_batch_effects. Default 'none'.
- **`batch_column`** *(optional)* — (str) - Metadata column that identifies independent acquisition batches, normally 'plateID'. Every analyzed row must have a value and at least batch_min_samples rows must occur in each batch. Use an acquisition date or instrument ID only if that is the nuisance source you intend to remove. Default 'plateID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_column`** *(optional)* — (str or None) - Metadata column containing reference-control labels for control_center, normally 'columnID' for plate controls. It is ignored by center, zscore, robust_zscore, and none. Blank follows col_to_compare in Image UMAP or location_column in Classify (ML); regression defaults to 'columnID'. API: spacr.batch_correction.correct_batch_effects.
- **`batch_control_values`** *(optional)* — (str, number, list or None) - Reference/negative-control value(s) in batch_control_column used by control_center. Each plate needs at least batch_min_samples matching rows. Image UMAP falls back to neg and Classify (ML) to negative_control_id when this field is blank; regression requires an explicit value. Default varies by module. API: spacr.batch_correction.correct_batch_effects.
- **`batch_covariate_column`** *(optional)* — (str, list or None) - Metadata column(s) naming the biological effects ComBat must preserve, for example 'condition' or 'condition,timepoint'. ComBat estimates the batch effect from residuals after fitting these terms, so unlisted effects may be removed with the plate effect. Include every treatment effect that must remain in the corrected data. See spacr.batch_correction.correct_batch_effects. Default None.
- **`batch_combat_mean_only`** *(optional)* — (bool) - True corrects only the additive batch shift and leaves each batch's scale alone. Use it when the plates differ in level but not in spread, or when a batch has too few rows for a stable variance estimate. False (the default) corrects both location and scale, which is standard ComBat. Ignored by every method other than combat. API: spacr.batch_correction.correct_batch_effects.
- **`batch_min_samples`** *(optional)* — (int) - Minimum number of rows required in every batch, and minimum matching reference controls per batch for control_center. Correction stops with an actionable error below this threshold because a one- or two-object plate estimate is unstable. Default 3. API: spacr.batch_correction.correct_batch_effects.
- **`batch_missing_control`** *(optional)* — (str) - Policy when control_center cannot find enough reference controls on a plate: 'error' stops rather than silently mixing corrected and raw plates; 'skip' leaves that plate unchanged and records a warning. Default 'error'. API: spacr.batch_correction.correct_batch_effects.

</details>

<details>
<summary>Response (5 settings)</summary>

#### Response

- **`dependent_variable`** *(required)* — (str) - Name of the column in score_data that is modelled as the response, e.g. 'pred'/'predictions' from the ML scoring step or a measured feature such as 'pathogen_nucleus_shortest_distance'. It is aggregated per well by agg_type and then optionally transformed. The run aborts if the column is absent from the score CSV. Default 'pred'.
- **`invert_dependent_variable`** *(optional)* — (bool or int) - Transform the response before per-well aggregation when lower scores represent a stronger phenotype. False or 0 leaves the response unchanged, True or 1 uses 1 - x (appropriate for probabilities), and -1 uses 1 / x (appropriate for distances or counts). Any other value raises ValueError in process_scores. The transformation changes coefficient signs and therefore the side of the volcano plot on which significant effects appear. Default False.
- **`analysis_unit`** *(optional)* — (str) - What one row of the model is. 'well' collapses each well's objects into a single value with agg_type first, so the well is the independent unit and the number of cells behind it only affects precision. 'cell' regresses the individual objects instead, which keeps power but treats cells from one well as independent when they are not, so standard errors are optimistic unless the model accounts for the clustering (regression_type='mixed'). This is the explicit spelling of agg_type=None, which used to change the unit of analysis silently. Default 'well'.
- **`agg_type`** *(optional)* — (str) - How per-object scores are collapsed to one value per well before regression: 'mean', 'median', 'quantile' (75th percentile), or None to skip aggregation and regress on individual objects. Median resists a handful of extreme cells; None keeps power but ignores within-well correlation. Forced to a per-well sum for poisson and to None for quantile. Default 'mean'.
- **`transform`** *(optional)* — (str or None) - Optional transform applied to the aggregated per-well response before fitting: 'log' (log1p), 'sqrt', 'square', 'beta' (logit for a proportional response, with endpoints moved away from 0 and 1 and reported in the summary), or None. Use a transform when the response is skewed and fails the normality check. The fit then reports coefficients for '&lt;transform&gt;_&lt;dependent_variable&gt;'. Default None. Regression starts at 'log', so its first fit applies log1p unless this is changed.

</details>

<details>
<summary>Model &amp; Inference (17 settings)</summary>

#### Model & Inference

- **`inference`** *(optional)* — (str) - How effects are tested; the readable front end for analysis_mode. Default 'nonparametric': each guide is a plate-blocked Freedman-Lane permutation with an empirical P, valid however many guides there are, but no P can be below 1/(guide_permutations + 1). 'parametric' fits every guide at once in the chosen regression_type, so it needs more wells than guides or no coefficient is identifiable. 'auto' counts guides and wells and takes the simultaneous fit only when the design supports it.
- **`analysis_mode`** *(optional)* — (str) - 'regression' fits the selected simultaneous model. 'guide_permutation' tests each guide as a plate-adjusted marginal association using blocked Freedman--Lane permutations and then corrects the requested support family. This setting is normally derived from inference; set it directly only to override that choice. Default 'regression'. The Regression module starts with inference='nonparametric', so its resolved initial mode is 'guide_permutation'.
- **`regression_type`** *(optional)* — (str) - Model family. Default 'mixed' nests guides inside genes as random effects, so guides disagreeing widens that gene's interval; it answers both levels at once, which is why 'level' greys out. Otherwise: 'ols', 'wls', 'rlm', 'huber' or 'quantile' for a continuous response; 'logit', 'probit', 'beta' or 'quasi_binomial' for fractions; 'poisson' for counts; 'lasso', 'ridge', 'elasticnet' or 'horseshoe' when the predictors outnumber the wells. Note regression_type 'beta' fits a beta GLM, while transform 'beta' only transforms the response.
- **`regression_backend`** *(optional)* — (str) - Selects the library and device that fit the chosen regression_type. 'statsmodels (CPU)' preserves the established results and is the default. 'torch (GPU)' accelerates mixed models, 'pyfixest (CPU)' accelerates OLS/WLS with absorbed plate-position effects, and 'glum (CPU)' accelerates wide GLMs. The selector describes measured costs and numerical differences for each option; unavailable or incompatible backends are greyed out with the reason. Default 'statsmodels (CPU)'.
- **`level`** *(optional)* — (str) - Result level. For regression, 'both' writes results_grna.csv and results_gene.csv and corrects each family separately; 'grna' reports guide effects, and 'gene' pools guides by gene. Nonparametric inference also honours this choice. Mixed models disable it because they estimate gene effects with guides nested inside genes. For proportion plots, the same key selects 'object', 'well', or 'plate' aggregation. Default 'both' for regression and 'object' for proportions.
- **`model_data_layout`** *(optional)* — (str) - Shape handed to a fixed-effects estimator. 'long' preserves the historical repeated well-guide formula; 'wide' pivots guide or gene fractions to one row per independent well before fitting. Mixed models and Freedman-Lane permutation testing require the long representation and convert wide input back to long automatically. Default 'long'.
- **`intercept`** *(optional)* — (str) - Definition of the fitted intercept. 'fitted' estimates it from the data as the predicted response when every predictor is at its reference level; in a one-hot gene design, this reference is the gene omitted by patsy and may not have a useful biological interpretation. 'control' subtracts the negative controls' mean response before fitting and suppresses the intercept term, so every coefficient is a difference from the controls. 'zero' fits through the origin. 'value' fixes the intercept at intercept_value. Default 'fitted'.
- **`intercept_value`** *(optional)* — (float) - The number the intercept is pinned to when intercept is 'value'. The response is shifted by it and the term suppressed, so the fit is exactly y = intercept_value + terms. Read for no other intercept setting, and the panel greys it out for them. Default 0.0.
- **`model_plate_position`** *(optional)* — (bool) - Include plateID, rowID, and columnID terms to model plate and spatial effects. random_row_column_effects selects fixed terms or a plate-grouped mixed model with row/column variance components. Enabling random effects requires this setting. In the reference screen the layout terms were jointly significant at p = 6.7e-23, so enable them when plate, edge, row, or column effects are plausible. Default False.
- **`random_row_column_effects`** *(optional)* — (bool) - Fit plate, row and column as random effects instead of fixed ones: True overrides regression_type to 'mixed' and fits a MixedLM grouped by plateID with rowID and columnID variance components, dropping them from the fixed-effect formula. Use it when edge or row artefacts differ between plates; it is slower and may fail to converge. Default False.
- **`multiple_testing_method`** *(optional)* — (str) - Correction applied within each outcome/support family: fdr_bh (Benjamini--Hochberg, default), fdr_by, bonferroni, holm, or none. Stricter family-wise methods generally call fewer guides.
- **`fdr_alpha`** *(optional)* — (float) - Family-level rejection threshold for adjusted P values in guide_permutation mode. Must be between 0 and 1. Default 0.05.
- **`p_threshold_alpha`** *(optional)* — (float) - P-value threshold used to call a coefficient a hit and to draw the volcano-plot reference line. It applies to the P-value type selected by p_threshold_kind, keeping results_significant.csv and the corresponding figure consistent. Supply a fraction strictly between 0 and 1; 5 is rejected as an invalid representation of 5%. Default 0.05.
- **`p_threshold_kind`** *(optional)* — (str) - Whether p_threshold_alpha is applied to the multiple-testing-adjusted p-value ('adjusted') or the raw per-coefficient p-value ('raw'). The analysis run and volcano plot use the same choice, so exported hits and plotted calls remain consistent. 'raw' generally calls more genes in screens with thousands of guides. Other values are rejected. Default 'adjusted'.
- **`threshold_method`** *(optional)* — (str) - Select the spread estimator for the control-based effect-size cutoff: 'std', legacy 'var' (squared units), 'mad', 'iqr', 'percentile' (the 95th percentile of absolute coefficients), or 'range'. 'none' disables the effect-size cutoff. Historical aliases such as 'standard_deveation', 'variance', and 'quantile' are accepted. Used only when controls are set. Default 'std'.
- **`threshold_multiplier`** *(optional)* — (float) - Set how many control-distribution spreads are required for a hit. The cutoff is abs(median(control coefficients)) + threshold_multiplier × spread, using threshold_method for the spread. Larger values demand a larger effect; threshold_method='none' disables the cutoff. Used only when controls are set. Default 3.
- **`annotation_source`** *(optional)* — (str) - Organism annotation joined to regression results. 'toxoplasma' uses bundled Toxoplasma gondii tables without a network. Other organism names or NCBI taxon IDs fetch UniProt entries; a single accession such as P04637 fetches that entry. Results are cached beside the outputs for offline reuse. An unrecognised name leaves results unannotated and suggests similar names. Leave empty for no annotation. Default 'toxoplasma'.

</details>

<details>
<summary>Estimator Tuning (14 settings)</summary>

#### Estimator Tuning

- **`cov_type`** *(optional)* — (str) - Heteroscedasticity-robust covariance estimator passed to likelihood fits: 'HC0', 'HC1', 'HC2', 'HC3', or None for classical non-robust errors. It changes standard errors and P-values, not coefficients. Use 'HC3' when residual variance increases with well cell count. Penalized, robust and quantile fits do not support this estimator and raise an error rather than reporting ordinary errors under a robust label. Default None. Read by regression_type 'glm', 'logit', 'ols', 'poisson', 'probit', 'quasi_binomial', 'spline', 'wls'.
- **`alpha`** *(optional)* — (float) - Regularisation strength for penalised models only: the L1 penalty for 'lasso', the L2 penalty for 'ridge', the combined penalty for 'elasticnet', and the inverse margin for 'hinge'. Larger values shrink more coefficients toward zero. Set it to 'auto' or None to select the value by five-fold cross-validation; the default 1 may over-regularise fraction-scale designs. Other model families reject a non-default alpha rather than ignoring it. Default 1. Read by regression_type 'elasticnet', 'hinge', 'lasso', 'ridge'.
- **`l1_ratio`** *(optional)* — (float) - How the elastic-net penalty is split between L1 and L2: 1.0 is a pure lasso (sparse, picks one gRNA out of a correlated group), 0.0 is a pure ridge (dense, shares the effect across the group), and values between keep some of both. Use 0.5 when correlated gRNAs of the same gene should be selected together rather than arbitrarily. Read only by regression_type 'elasticnet'. Default 0.5.
- **`quantile`** *(optional)* — (float) - Which quantile of the response quantile regression fits, strictly inside 0 and 1: 0.5 is the median (robust to outlier wells), 0.9 asks which gRNAs move the top of the distribution rather than its centre. Aggregation is turned off automatically so the quantile is taken over cells, not over well means. Read only by regression_type 'quantile'; it replaced the old overload of alpha. Default 0.5.
- **`huber_t`** *(optional)* — (float) - Where Huber's loss switches from squared to linear, in units of the estimated residual scale, for the robust fits. Smaller values downweight more wells and resist heavier contamination; larger values approach ordinary least squares. The default 1.345 gives 95 percent of the efficiency of OLS under normally distributed residuals. Read only by regression_type 'rlm' and 'huber'. Default 1.345.
- **`spline_knots`** *(optional)* — (int) - Number of knots in the spline basis of each continuous covariate. More knots let the fitted curve bend more freely and use more degrees of freedom; a covariate with fewer distinct values than the degree is left linear and gets no spline basis. The guide columns are never given a basis, so the fit still returns one coefficient and one p-value per guide. Read only by regression_type 'spline'. Default 4.
- **`spline_degree`** *(optional)* — (int) - The polynomial degree of each covariate's spline basis. 3 is a cubic spline, the usual choice; 1 is piecewise linear. Read only by regression_type 'spline'. Default 3.
- **`hinge_threshold`** *(optional)* — (float) - Response value above which a well counts as positive for the hinge (linear SVM) fit. Leave it None when the response is already binary, in which case the two values it holds become the two classes. spaCR refuses a continuous response with no threshold rather than splitting it at the mean or median, because a cut chosen by the software decides the hypothesis being tested. Read only by regression_type 'hinge'. Default None.
- **`hinge_n_boot`** *(optional)* — (int) - Number of bootstrap resamples behind the hinge p-values. A support vector machine has no likelihood and so no Wald test; spaCR refits it on this many resamples of the wells and compares each coefficient to its bootstrap standard deviation. Treat the result as a stability statistic, not a hypothesis test. Higher is steadier and linearly slower; below about 50 the standard deviations are too noisy to rank on. Default 200. Read by regression_type 'hinge'.
- **`lasso_n_boot`** *(optional)* — (int) - Number of bootstrap resamples used to rank lasso and elastic-net hits by how often each gRNA survives the penalty. These models have no valid p-values, so selection frequency replaces the significance test entirely. Higher is steadier and linearly slower; the cost is one full penalised fit per resample, doubled when alpha is 'auto' because each resample cross-validates. Default 200. Read by regression_type 'elasticnet', 'group_lasso', 'lasso'.
- **`lasso_selection_threshold`** *(optional)* — (float) - Minimum bootstrap selection frequency, between 0 and 1, for a lasso or elastic-net coefficient to be called a hit. 0.6 means the gRNA kept a non-zero coefficient in at least three fifths of the resamples. Raise it for a shorter list of more consistently selected terms; lowering it below about 0.5 admits terms the penalty drops as often as it keeps. Default 0.6. Read by regression_type 'elasticnet', 'group_lasso', 'lasso'.
- **`group_lasso_lambda`** *(optional)* — (float) - Penalty weight of the group lasso, which shrinks all of one gene's guides together rather than one at a time, so a gene enters or leaves the model as a unit instead of on its single strongest guide. Larger values keep fewer genes; 0 leaves the fit unpenalised and negative is refused. Set it to auto to choose it by cross-validation. Default auto. Read by regression_type 'group_lasso'.
- **`rra_alpha`** *(optional)* — (float) - The top fraction of the ranked guide list robust rank aggregation scores against: 0.25 asks whether a gene's guides cluster in the best quarter of the ranking more than chance allows, ignoring the rest. Smaller is stricter and returns fewer, better-supported genes. Above 0 and at most 1; 25 for '25%' is refused. Default 0.25. Read by regression_type 'rra'.
- **`rra_permutations`** *(optional)* — (int) - How many permuted rankings the robust rank aggregation null is built from. The smallest P value it can report is about 1/rra_permutations, so 10000 resolves the tail to 1e-4; raise it when many genes pile up at that floor and lower it while exploring, since the cost is linear in this number. Default 10000. Read by regression_type 'rra'.

</details>

<details>
<summary>Permutation Test (9 settings)</summary>

#### Permutation Test

- **`grna_statistic`** *(optional)* — (str) - What the permutation test measures between a gRNA's well fractions and the well phenotype. 'pearson' is a partial correlation, which is linear and is moved by an extreme well in proportion to how extreme it is. 'rank' is the same quantity computed on the ranked phenotype, so it responds to order rather than magnitude and no single well can move it far. Both cost one matrix product, so the choice does not change how long the test takes. Default 'pearson'.
- **`guide_min_wells`** *(optional)* — (int or list) - Minimum numbers of independent wells containing a guide. A list such as [1, 2, 3, 4] writes one sensitivity-analysis table and volcano plot per threshold; P values are computed once and the multiple-testing correction is repeated within each eligible family. Default [1, 2, 3, 4].
- **`guide_primary_min_wells`** *(optional)* — (int or None) - Which guide_min_wells family supplies results_significant.csv and the returned 'significant' table. Default None chooses the smallest requested threshold.
- **`guide_permutations`** *(optional)* — (int) - Number of plate-blocked Freedman--Lane residual permutations used for empirical two-sided guide P values. The estimator is (exceedances + 1) / (permutations + 1), where exceedances are permuted statistics at least as extreme as the observed statistic; this imposes a hard floor of 1 / (permutations + 1). Values of 1,000, 10,000, and 200,000 resolve floors of approximately 1e-3, 1e-4, and 5e-6, respectively. Increase the count when results accumulate at this resolution floor; runtime increases linearly. Default 200000.
- **`guide_permutation_seed`** *(optional)* — (int) - Random seed for reproducible residual permutations. Keep it fixed to reproduce exact empirical P values; change it to check Monte Carlo sensitivity. Default 0.
- **`guide_permutation_block`** *(optional)* — (str) - Column defining exchangeability blocks for permutations, normally plateID. Residuals are never shuffled between its levels. Default 'plateID'.
- **`guide_nuisance_columns`** *(optional)* — (list) - Additional measured well-level covariates to residualize from both phenotype and guide fraction before testing. Do not put post-treatment outcomes here. Default []. Regression starts with ['rowID', 'columnID'] to remove plate-position structure before the within-plate permutation.
- **`guide_presence_threshold`** *(optional)* — (float) - A guide counts as present in a well only when its fraction is above this value. The effect still uses the unthresholded fraction. Default 0.0.
- **`guide_permutation_batch_size`** *(optional)* — (int) - Number of permutation outcomes evaluated together. Lower this if memory is tight; it does not change the result. Default 500.

</details>

<details>
<summary>Additional settings (10 settings)</summary>

#### Additional settings

- **`analysis_excluded_wells`** *(optional)* — (list or None) - Wells dropped before anything is fitted: the sequencing sweep removes them from the count table before it looks for the fraction threshold, and the regression never sees them. It must name the same wells as filter_value or the threshold is fitted on wells the fit has already dropped. Regression initializes it from filter_value plus any declared control blocks (the shipped default is ['c1', 'c2', 'c3']). Set a column ('c12'), row ('r1'), well ('r1_c12'), or complete plate key.
- **`cell_area_outlier_mads`** *(optional)* — (float | None) - Optionally remove objects whose cell area exceeds this many scaled median absolute deviations from the median. Filtering occurs before guide fractions are computed, so removed objects do not contribute to normalization. MAD-based limits are robust to skewed area distributions. Set to None to disable; a value of 5 is a conservative starting point. Default None.
- **`cell_intensity_outlier_mads`** *(optional)* — (float | None) - Optionally apply the scaled-MAD filter to cell-channel intensity before guide annotation. This can exclude extreme measurements from overexposure or bright debris. Set to None to disable. Default None.
- **`max_failure_rate`** *(optional)* — (float or None) - Fraction of failed items above which the run aborts. For example, 0.2 aborts after more than 20% of items fail. The failure ledger is written to the artifact before the abort. None disables rate-based abortion; failures remain counted and reported, and incomplete artifacts are marked partial. Default None.
- **`nucleus_area_outlier_mads`** *(optional)* — (float | None) - Scaled median-absolute-deviation threshold for nucleus area before guide annotation. Lower values remove more nucleus-size extremes before guide fractions and normalization are computed. Set to None to disable. Default None.
- **`nucleus_intensity_outlier_mads`** *(optional)* — (float | None) - Scaled median-absolute-deviation threshold for nucleus-channel intensity before guide annotation. Lower values remove more extreme measurements, including saturated nuclei or bright debris. Set to None to disable. Default None.
- **`regression_panel_manifest`** *(optional)* — (dict, str, or None) - Publication panels built after the regression CSVs are written. None skips them. Supply a mapping, or a path to JSON/YAML, declaring each panel's result source, phenotype, gene/gRNA level and plot kind; the packages land under publication_panels/. Structured rather than a one-line field, so it is set from a script or a saved settings file rather than in the GUI. Default None.
- **`regression_qc`** *(optional)* — (bool) - Write variance-homogeneity, residual, design, influence, and calibration diagnostics to &lt;res_folder&gt;/regression_qc/ as figures, a combined PDF, and a text report. One fit requires approximately 5.8 seconds and writes 19 files, so this is enabled for individual analyses but disabled automatically during parameter sweeps to avoid producing thousands of diagnostic files. Reopen a selected trial to generate its diagnostics. Applies to every regression_type. Default True.
- **`strict_errors`** *(optional)* — (bool or None) - Error-handling policy for recoverable steps. Off records failures in the run ledger and final summary while continuing with successful items. On raises immediately for setup or configuration errors such as unreadable paths, missing columns or inaccessible databases, preventing partial batch results from invalid inputs. Per-item failures such as one corrupt image remain recoverable under either policy. None defers to $SPACR_STRICT_ERRORS. Default None.
- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression. Invasion and Replication also start with console detail disabled.

</details>

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Input Tables
    # Required settings
    'score_data': [],
    'count_data': [],
    'paired_data': [],
    # Optional settings
    'src': '',

    # Response
    # Required settings
    'dependent_variable': 'pred',
}

In [ ]:
settings.update({
    # Input Tables
    # Optional settings
    'metadata_files': [],
    'count_grna_column': 'grna',
    'count_value_column': 'count',
    'independent_variable_layout': 'auto',
    'wide_predictor_columns': [],

    # Controls & Filters
    # Optional settings
    'positive_control_id': '239740',
    'negative_control_id': '233460',
    'positive_control_wells': None,
    'negative_control_wells': None,
    'mixed_control_wells': None,
    'exclude_grnas': None,
    'nontargeting_control_grnas': ['000000'],
    'filter_column': 'columnID',
    'filter_value': ['c1', 'c2', 'c3'],
    'min_cells_per_well': 100,
    'min_observations_per_hit': 0,
    'fraction_threshold': 0.02,
    'calibrate_fraction_threshold': False,
    'normalise_fraction': True,
    'target_unique_count': 5,
    'tolerance': 0.02,
    'outlier_detection': True,

    # Plate & Batch Correction
    # Optional settings
    'batch_correction': 'none',
    'batch_column': 'plateID',
    'batch_control_column': 'columnID',
    'batch_control_values': None,
    'batch_covariate_column': None,
    'batch_combat_mean_only': False,
    'batch_min_samples': 3,
    'batch_missing_control': 'error',

    # Response
    # Optional settings
    'invert_dependent_variable': False,
    'analysis_unit': 'well',
    'agg_type': 'mean',
    'transform': 'log',

    # Model & Inference
    # Optional settings
    'inference': 'nonparametric',
    'analysis_mode': 'guide_permutation',
    'regression_type': 'mixed',
    'regression_backend': 'statsmodels (CPU)',
    'level': 'both',
    'model_data_layout': 'long',
    'intercept': 'fitted',
    'intercept_value': 0.0,
    'model_plate_position': False,
    'random_row_column_effects': False,
    'multiple_testing_method': 'fdr_bh',
    'fdr_alpha': 0.05,
    'p_threshold_alpha': 0.05,
    'p_threshold_kind': 'adjusted',
    'threshold_method': 'std',
    'threshold_multiplier': 3,
    'annotation_source': 'toxoplasma',

    # Estimator Tuning
    # Optional settings
    'cov_type': None,
    'alpha': 1,
    'l1_ratio': 0.5,
    'quantile': 0.5,
    'huber_t': 1.345,
    'spline_knots': 4,
    'spline_degree': 3,
    'hinge_threshold': None,
    'hinge_n_boot': 200,
    'lasso_n_boot': 200,
    'lasso_selection_threshold': 0.6,
    'group_lasso_lambda': 'auto',
    'rra_alpha': 0.25,
    'rra_permutations': 10000,

    # Permutation Test
    # Optional settings
    'grna_statistic': 'pearson',
    'guide_min_wells': [1, 2, 3, 4],
    'guide_primary_min_wells': None,
    'guide_permutations': 200000,
    'guide_permutation_seed': 0,
    'guide_permutation_block': 'plateID',
    'guide_nuisance_columns': ['rowID', 'columnID'],
    'guide_presence_threshold': 0.0,
    'guide_permutation_batch_size': 500,

    # Additional settings
    # Optional settings
    'analysis_excluded_wells': ['c1', 'c2', 'c3'],
    'cell_area_outlier_mads': None,
    'cell_intensity_outlier_mads': None,
    'max_failure_rate': None,
    'nucleus_area_outlier_mads': None,
    'nucleus_intensity_outlier_mads': None,
    'regression_panel_manifest': None,
    'regression_qc': True,
    'strict_errors': None,
    'verbose': False,
})

In [ ]:
perform_regression(settings)

## Outputs and next steps

Effect estimates, uncertainty or resampling support, adjusted significance values, diagnostic plots, and ranked findings.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)